### Step 1: Setup Paths and Load Inputs
Resolves the input/output file locations relative to this notebook, loads the Step-1 filtered claims file, and loads the ICD-10 complications master list that powers the description-to-code lookup in Step 2.

In [8]:
import pandas as pd
import numpy as np
import os

# 1. SETUP PATHS
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, "..", "csv")

input_file = os.path.join(data_dir, "step1_claims_filtered.xlsx")
# We need the Master List to translate the Descriptions
path_comp_master = os.path.join(data_dir, "diabetes_list_of_complications.xlsx")
output_file = os.path.join(data_dir, "step2_claims_wide_format.xlsx")

print("--- STEP 2: RENAMING & MAPPING ---")

if not os.path.exists(input_file):
    print(" Error: Step 1 file missing.")
else:
    df = pd.read_excel(input_file)
    print(f" Loaded Claims: {len(df)} rows.")
    
    # Load Master List (It has the Description -> Code link)
    df_master = pd.read_excel(path_comp_master)
    df_master.columns = [str(c).strip() for c in df_master.columns]
    print(f" Loaded Master List: {len(df_master)} codes.")

--- STEP 2: RENAMING & MAPPING ---
✅ Loaded Claims: 315402 rows.
✅ Loaded Master List: 141 codes.


### Step 2: Advanced Mapping (handle multi-line descriptions)
Takes the free-text `OTHER_DIAGNOSIS` field (often containing multiple diagnoses separated by newlines) and pivots it into clean `DIAG_2`, `DIAG_3`, `DIAG_4` columns mapped to the ICD-10 master list.

The previous version merged the pivoted frame onto the base claims **while `DIAG_2` already existed on the left side from an earlier rename**, which made pandas auto-suffix both into `DIAG_2_x` and `DIAG_2_y` — identical values duplicated. This version drops the pre-existing `DIAG_2` before the merge (if present) so exactly one `DIAG_2` column survives.

In [ ]:
# 2. ADVANCED MAPPING (Handle Multi-Line Descriptions)
print("Mapping text descriptions to ICD-10 codes (Handling multiple lines)...")

# --- A. AUTO-DETECT MASTER COLUMNS ---
code_col = None
desc_col = None

for col in df_master.columns:
    c_lower = str(col).lower().strip()
    if "code" in c_lower and ("diabetes" in c_lower or "diag" in c_lower):
        code_col = col
    if "desc" in c_lower:
        desc_col = col

print(f" Detected Code Column: '{code_col}'")
print(f" Detected Desc Column: '{desc_col}'")

if code_col and desc_col:
    # 1. Create Dictionary
    map_dict = pd.Series(
        df_master[code_col].values,
        index=df_master[desc_col].astype(str).str.strip().str.lower()
    ).to_dict()

    # 2. PREPARE THE DATA (The "Explode" Strategy)
    # Step A: Clean formatting (remove brackets/quotes)
    df["clean_desc"] = df["OTHER_DIAGNOSIS"].astype(str).str.replace(r"[\[\]']", "", regex=True).str.strip().str.lower()

    # Step B: Split by newline (\n) to handle multiple diagnoses
    # This turns "Diag A \n Diag B" into a list ["Diag A", "Diag B"]
    df["desc_list"] = df["clean_desc"].str.split(r"\\n|\n")

    # Step C: Explode! (Create 1 row per description)
    # We only explode the relevant columns to save memory
    df_exploded = df[["CLAIM_CODE", "desc_list"]].explode("desc_list")

    # Step D: Trim whitespace from the split parts
    df_exploded["single_desc"] = df_exploded["desc_list"].str.strip()

    # 3. PERFORM MAPPING
    df_exploded["DIAG_CODE_MAPPED"] = df_exploded["single_desc"].map(map_dict)

    # Filter out nulls (things that didn't map or were empty strings)
    df_valid_diags = df_exploded.dropna(subset=["DIAG_CODE_MAPPED"])

    # 4. PIVOT BACK TO COLUMNS (DIAG_2, DIAG_3...)
    # Create a sequence counter (1, 2, 3...) for each claim
    # Start at 2 because DIAG_1 is taken by the primary code
    df_valid_diags["seq"] = df_valid_diags.groupby("CLAIM_CODE").cumcount() + 2

    # Pivot: Index=Claim, Col=Seq, Value=Code
    df_pivoted = df_valid_diags.pivot(index="CLAIM_CODE", columns="seq", values="DIAG_CODE_MAPPED")

    # Rename columns to DIAG_2, DIAG_3...
    df_pivoted.columns = [f"DIAG_{c}" for c in df_pivoted.columns]
    df_pivoted = df_pivoted.reset_index()

    print(f" Successfully extracted extra diagnoses!")
    print(f"   -> Columns created: {df_pivoted.columns.tolist()}")

    # 5. MERGE BACK TO MAIN DATA
    # Drop helper columns we don't need for the final table
    cols_to_keep = [c for c in df.columns if c not in ["clean_desc", "desc_list"]]
    # Deduplicate main table to get 1 row per claim
    df_clean = df[cols_to_keep].drop_duplicates(subset="CLAIM_CODE")

    # CRITICAL: drop any pre-existing DIAG_2 from df_clean before the merge.
    # If left in, pandas auto-suffixes the collision into DIAG_2_x / DIAG_2_y
    # (identical values, two columns) which pollutes the downstream schema.
    pre_existing_diag_cols = [c for c in df_clean.columns if c in df_pivoted.columns and c != "CLAIM_CODE"]
    if pre_existing_diag_cols:
        print(f"   -> Dropping pre-existing {pre_existing_diag_cols} from df_clean to avoid _x/_y suffixing on merge.")
        df_clean = df_clean.drop(columns=pre_existing_diag_cols)

    # Merge the new DIAG columns
    df_final_step2 = pd.merge(df_clean, df_pivoted, on="CLAIM_CODE", how="left")

    # Rename original Primary Code to DIAG_1
    if "DIAGNOSIS_CODE" in df_final_step2.columns:
        df_final_step2.rename(columns={"DIAGNOSIS_CODE": "DIAG_1"}, inplace=True)

    # Ensure DIAG_2 exists even if no claim had a secondary diagnosis
    if "DIAG_2" not in df_final_step2.columns:
        df_final_step2["DIAG_2"] = np.nan

    print(f" Final Shape: {df_final_step2.shape}")
    display(df_final_step2.head())

    # Set variable for saving
    df_wide = df_final_step2

else:
    print(" Error: Master List columns not found.")

### Step 3: Save the wide-format claims file
Writes the reshaped frame to `step2_claims_wide_format.xlsx`, which becomes the input to Step 3 (feature engineering).

In [13]:
# 3. SAVE TO EXCEL
if 'df_wide' in locals():
    print(f"💾 Saving {len(df_wide)} rows to: {output_file} ...")
    df_wide.to_excel(output_file, index=False)
    print(" Done. Step 2 Complete.")
else:
    print(" Error: Processing failed in previous step.")

💾 Saving 315402 rows to: c:\Users\thiranbarath\Documents\GitHub\dataPreprocessing\source code\..\csv\step2_claims_wide_format.xlsx ...
✅ Done. Step 2 Complete.
